Compare Vinardo Dumps

tool to compare `smina` and `muDock` CSV dumps in `benchmark/testx` based on the atom id




In [4]:
from __future__ import annotations

import csv
import pandas as pd
from pathlib import Path

BENCHMARK_ROOT = Path('/Users/tommasovaccari/Coding/projects/molecular-docking/muDock/benchmark')
FIELDS = ['atom_name', 'smina_type', 'xs_radius', 'xs_hydrophobe', 'xs_donor', 'xs_acceptor']
BENCHMARK_ROOT


PosixPath('/Users/tommasovaccari/Coding/projects/molecular-docking/muDock/benchmark')

In [5]:
def load_rows(path: Path) -> dict[int, dict[str, str]]:
    with path.open(newline='') as handle:
        rows = list(csv.DictReader(handle))
    by_id: dict[int, dict[str, str]] = {}
    for row in rows:
        atom_id = int(row['id'])
        if atom_id in by_id:
            raise ValueError(f'duplicate id {atom_id} in {path}')
        by_id[atom_id] = row
    return by_id


def compare_csv_pair(left: Path, right: Path) -> dict:
    left_rows = load_rows(left)
    right_rows = load_rows(right)

    left_ids = set(left_rows)
    right_ids = set(right_rows)
    missing_in_right = sorted(left_ids - right_ids)
    missing_in_left = sorted(right_ids - left_ids)

    mismatches = []
    for atom_id in sorted(left_ids & right_ids):
        for field in FIELDS:
            if left_rows[atom_id][field] != right_rows[atom_id][field]:
                mismatches.append({
                    'id': atom_id,
                    'field': field,
                    'smina': left_rows[atom_id][field],
                    'mudock': right_rows[atom_id][field],
                })

    return {
        'missing_in_mudock': missing_in_right,
        'missing_in_smina': missing_in_left,
        'mismatches': mismatches,
    }


In [7]:
def compare_case(case_dir: Path) -> dict:
    protein = compare_csv_pair(case_dir / 'smina' / 'protein_atoms.csv', case_dir / 'mudock' / 'protein_atoms.csv')
    ligand = compare_csv_pair(case_dir / 'smina' / 'ligand_atoms.csv', case_dir / 'mudock' / 'ligand_atoms.csv')
    return {'protein': protein, 'ligand': ligand}


def summarize(root: Path) -> dict[str, dict]:
    results = {}
    for case_dir in sorted([p for p in root.iterdir() if p.is_dir() and p.name.startswith('test')], key=lambda p: int(p.name[4:])):
        results[case_dir.name] = compare_case(case_dir)
    return results

results = summarize(BENCHMARK_ROOT)
summary = {
    case: {
        'protein_mismatches': len(data['protein']['mismatches']),
        'ligand_mismatches': len(data['ligand']['mismatches']),
        'protein_missing': len(data['protein']['missing_in_mudock']) + len(data['protein']['missing_in_smina']),
        'ligand_missing': len(data['ligand']['missing_in_mudock']) + len(data['ligand']['missing_in_smina']),
    }
    for case, data in results.items()
}
summary_df = pd.DataFrame.from_dict(summary, orient='index')
summary_df.index.name = 'case'
summary_df = summary_df.sort_index(key=lambda index: index.str[4:].astype(int))
summary_df


,protein_mismatches,ligand_mismatches,protein_missing,ligand_missing
case,,,,
test1,0,0,0,0
test2,0,0,0,0
test3,0,0,0,0
test4,0,0,0,0
test5,0,0,0,0
test6,0,0,0,0
test7,0,0,0,0
test8,0,0,0,0
test9,0,0,0,0


Seems al good for now, the next step is to introduce the testing for the atom preprocesssing.